In [ ]:
import pandas as pd
import numpy as np

: 

In [ ]:
# 1. BACA DATA NASA POWER
nasa_file = "POWER_Point_Daily_20200101_20261115_001d29S_113d41E_LST.csv" 

with open(nasa_file, "r") as f:
    for i, line in enumerate(f):
        if "-END HEADER-" in line:
            skip_rows = i + 1
            break

df_cuaca = pd.read_csv(nasa_file, skiprows=skip_rows)
df_cuaca.replace(-999, np.nan, inplace=True)
df_cuaca.dropna(inplace=True)

df_cuaca["date"] = pd.to_datetime(
    df_cuaca[["YEAR", "MO", "DY"]].rename(
        columns={"YEAR": "year", "MO": "month", "DY": "day"}
    )
).dt.strftime("%Y-%m-%d")

df_cuaca = df_cuaca[["date", "T2M", "RH2M", "PRECTOTCORR", "GWETTOP"]]

In [ ]:
# 2. BACA & FILTER DATA HOTSPOT (Sesuai kolom daily_province.csv)
df_hotspot = pd.read_csv("daily_province.csv")

# Ganti sesuai provinsi yang titik koordinatnya kamu ambil di NASA POWER
nama_provinsi = "Central Kalimantan"

df_hotspot_prov = df_hotspot[df_hotspot["PROVINCE"] == nama_provinsi].copy()

In [ ]:
# Jumlahkan hotspot dari satelit SNPP dan NOAA20
df_hotspot_prov["TOTAL_HOTSPOTS"] = (
    df_hotspot_prov["HOTSPOTS_SNPP"].fillna(0) +
    df_hotspot_prov["HOTSPOTS_NOAA20"].fillna(0)
)

# Parse tanggal format DD/MM/YYYY
df_hotspot_prov["date"] = pd.to_datetime(
    df_hotspot_prov["DATE"], dayfirst=True
).dt.strftime("%Y-%m-%d")

df_hotspot_prov = df_hotspot_prov[["date", "TOTAL_HOTSPOTS"]]

In [ ]:
# 3. GABUNGKAN (LEFT JOIN) BERDASARKAN TANGGAL
df_final = pd.merge(df_cuaca, df_hotspot_prov, on="date", how="left")
df_final["TOTAL_HOTSPOTS"] = df_final["TOTAL_HOTSPOTS"].fillna(0).astype(int)

In [ ]:
# 4. PELABELAN MULTI-KELAS (Aman, Waspada, Bahaya)
def tentukan_risiko(jumlah):
    if jumlah == 0:
        return "Aman"
    elif jumlah <= 10:
        return "Waspada"
    else:
        return "Bahaya"

df_final["Status_Risiko"] = df_final["TOTAL_HOTSPOTS"].apply(tentukan_risiko)

In [ ]:
df_final.rename(columns={
    "T2M": "Suhu_Udara",
    "RH2M": "Kelembapan_Udara",
    "PRECTOTCORR": "Curah_Hujan",
    "GWETTOP": "Kelembapan_Tanah"
}, inplace=True)

print(df_final.head(10))
print("\nDistribusi Kelas Risiko:")
print(df_final["Status_Risiko"].value_counts())

In [ ]:
# Simpan menjadi dataset siap latih
df_final.to_csv("dataset_karhutla_siap_latih.csv", index=False)